# Project 2: Specialized Zero-Shot Medical Chart & Prescription Intelligence
### Complete AI Server: `Qwen/Qwen2.5-VL-7B-Instruct` on Google Colab T4 GPU

**Quick Guide:**
1. In Google Colab top menu, go to **Runtime** > **Change runtime type** > Select **T4 GPU**.
2. Run **Cell 1** (Installs packages and automatically restarts the session).
3. Run **Cell 2** (Loads Qwen2.5-VL into GPU VRAM and starts the public Cloudflare tunnel server).
4. *(Optional)* Run **Cell 3** to upload an image and print the report directly inside Google Colab!

In [ ]:
# CELL 1: Install Dependencies (Runs in ~30s and reboots runtime automatically)
!pip install -q -U --force-reinstall "pillow>=10.4.0,<11.2.0"
!pip install -q -U "bitsandbytes>=0.46.1" "transformers>=4.49.0" accelerate torchvision
!pip install -q -U qwen-vl-utils fastapi uvicorn nest_asyncio pydantic python-multipart

print("\n✅ Dependencies installed successfully! Rebooting runtime now for clean CUDA linkage...")
import os
os.kill(os.getpid(), 9)

In [ ]:
# CELL 2: All-in-One Master Server (Model Loading + FastAPI Server + Cloudflare Tunnel)
import os
import io
import re
import json
import time
import base64
import subprocess
import nest_asyncio
import uvicorn
import torch
from PIL import Image
from typing import List, Optional
from pydantic import BaseModel
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

nest_asyncio.apply()
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# ---------------------------------------------------------------------------
# 1. LOAD MODEL INTO T4 GPU VRAM (4-bit NF4 Quantization)
# ---------------------------------------------------------------------------
print("=" * 75)
print(f"📥 Loading {MODEL_ID} onto T4 GPU...")
print("=" * 75)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=1280 * 28 * 28
)

gpu_vram = torch.cuda.memory_allocated() / 1e9
print(f"\n✅ Model loaded successfully into GPU VRAM!")
print(f"🔥 GPU VRAM in use: {gpu_vram:.2f} GB / 15.00 GB")
print("=" * 75 + "\n")

# ---------------------------------------------------------------------------
# 2. DEFINE CLINICAL PROMPT & FASTAPI APPLICATION
# ---------------------------------------------------------------------------
app = FastAPI(title="Qwen2.5-VL Clinical Intelligence Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

CLINICAL_SYSTEM_PROMPT = """You are a board-certified Clinical Document AI and Medical Chart Intelligence system.
Your mission is to perform rigorous, zero-shot transcription and laboratory interpretation from photographs of messy doctor prescriptions and complex diagnostic lab reports.

RULES:
1. Examine handwriting with utmost care: decipher medication names (brand vs generic), dosage strengths (e.g., 500mg, 40mg, 60K IU), dosage forms (Tablet, Capsule, Syrup, Sachet, Injection), and schedule notation (1-0-1, OD, BD, TDS, QID, HS, Before Food, After Food).
2. For laboratory tables, extract test names (e.g., Vitamin D, Hemoglobin, Fasting Blood Sugar, HbA1c, Serum Creatinine), numeric observed values, units, and reference intervals.
3. Classify each lab marker status as NORMAL, HIGH, LOW, CRITICAL_HIGH, or CRITICAL_LOW.
4. If an observed metric is outside safe boundaries, flag is_out_of_range = true and assign appropriate risk_level (moderate, high, critical).
5. Generate high-priority clinical alerts for severe anomalies or critical medication precautions.

OUTPUT FORMAT:
You MUST respond with ONLY a strictly valid JSON object adhering to this schema:
{
  "document_type": "handwritten_prescription" | "laboratory_report" | "composite_chart",
  "patient_info": {
    "name": "string",
    "age": "string",
    "gender": "string",
    "date": "string",
    "uhid_patient_id": "string"
  },
  "doctor_hospital_info": {
    "doctor_name": "string",
    "clinic_hospital": "string",
    "specialty": "string",
    "contact": "string"
  },
  "diagnosis_notes": ["string"],
  "medications": [
    {
      "drug_name": "string",
      "generic_name": "string",
      "dosage": "string",
      "form": "Tablet" | "Capsule" | "Syrup" | "Sachet" | "Drops" | "Injection" | "Other",
      "timing": {
        "morning": true/false,
        "afternoon": true/false,
        "evening": true/false,
        "night": true/false
      },
      "food_relation": "Before Food" | "After Food" | "With Food" | "Empty Stomach" | "As Needed",
      "duration": "string",
      "instructions": "string",
      "confidence": "high" | "medium" | "low"
    }
  ],
  "lab_metrics": [
    {
      "test_name": "string",
      "category": "string",
      "observed_value": "string or number",
      "unit": "string",
      "reference_interval": {
        "min": number or null,
        "max": number or null,
        "text": "string"
      },
      "status": "NORMAL" | "HIGH" | "LOW" | "CRITICAL_HIGH" | "CRITICAL_LOW",
      "is_out_of_range": true/false,
      "risk_level": "none" | "moderate" | "high" | "critical",
      "clinical_significance": "string"
    }
  ],
  "critical_alerts": ["string"],
  "transcription_confidence_summary": "string"
}
"""

class AnalyzeRequest(BaseModel):
    images: List[str]
    temperature: Optional[float] = 0.1
    max_tokens: Optional[int] = 3072

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model_name": MODEL_ID,
        "gpu_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
        "vram_allocated_gb": round(torch.cuda.memory_allocated() / 1e9, 2)
    }

@app.post("/analyze")
def analyze(req: AnalyzeRequest):
    if not req.images:
        raise HTTPException(status_code=400, detail="No images supplied")
    
    pil_images = []
    for idx, b64_str in enumerate(req.images):
        if "," in b64_str:
            b64_str = b64_str.split(",", 1)[1]
        img_bytes = base64.b64decode(b64_str)
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        pil_images.append(img)
    
    content_list = []
    for img in pil_images:
        content_list.append({"type": "image", "image": img})
    content_list.append({
        "type": "text",
        "text": "Analyze this medical document in detail. Extract patient details, transcribe doctor handwriting, build the medication dosage timing matrix, read all laboratory markers, evaluate reference intervals, and return the strict JSON schema."
    })

    messages = [
        {"role": "system", "content": CLINICAL_SYSTEM_PROMPT},
        {"role": "user", "content": content_list}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=req.max_tokens,
            temperature=req.temperature,
            do_sample=False
        )
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]

    clean_text = output_text.strip()
    if clean_text.startswith("```json"):
        clean_text = clean_text[7:]
    elif clean_text.startswith("```"):
        clean_text = clean_text[3:]
    if clean_text.endswith("```"):
        clean_text = clean_text[:-3]
    clean_text = clean_text.strip()

    json_match = re.search(r"(\{.*\})", clean_text, re.DOTALL)
    if json_match:
        clean_text = json_match.group(1)

    try:
        return json.loads(clean_text)
    except Exception as err:
        return {
            "document_type": "composite_chart",
            "transcription_confidence_summary": output_text,
            "patient_info": {},
            "doctor_hospital_info": {},
            "diagnosis_notes": ["Transcribed with minor format divergence."],
            "medications": [],
            "lab_metrics": [],
            "critical_alerts": [],
            "raw_text": output_text
        }

# ---------------------------------------------------------------------------
# 3. LAUNCH CLOUDFLARE TUNNEL & START SERVER
# ---------------------------------------------------------------------------
print("⚡ Starting Free Cloudflare Tunnel (Zero signup / No token required)...\n")
subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=False)
subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

cf_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = cf_process.stderr.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("=" * 75)
print("🚀 YOUR PUBLIC COLAB SERVER URL IS LIVE & READY:")
print(f"   {public_url}")
print("=" * 75)
print(f"👉 Copy this URL and paste it into your laptop dashboard header or backend/.env:")
print(f"   COLAB_API_URL={public_url}")
print("=" * 75 + "\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()


In [ ]:
# CELL 3: Optional - Test an Image Directly in Colab (Prints Clinical Report in 15 seconds)
from google.colab import files
print("Select a prescription or lab report image to analyze right here in Colab:")
uploaded_files = files.upload()

if uploaded_files:
    img_name = list(uploaded_files.keys())[0]
    print(f"\nAnalyzing {img_name} with Qwen2.5-VL-7B...")
    
    test_messages = [
        {"role": "system", "content": CLINICAL_SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "image", "image": img_name},
            {"type": "text", "text": "Analyze this medical document. Extract patient info, doctor prescriptions, dosage schedule (1-0-1), and lab table metrics. Return valid JSON."}
        ]}
    ]
    
    text_in = processor.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
    img_in, vid_in = process_vision_info(test_messages)
    model_inputs = processor(text=[text_in], images=img_in, videos=vid_in, padding=True, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        output_tokens = model.generate(**model_inputs, max_new_tokens=2048, do_sample=False)
        trimmed_tokens = [o[len(i):] for i, o in zip(model_inputs.input_ids, output_tokens)]
        report_text = processor.batch_decode(trimmed_tokens, skip_special_tokens=True)[0]
    
    print("\n" + "=" * 75)
    print("📋 CLINICAL ZERO-SHOT EXTRACTION REPORT (Generated in Colab):")
    print("=" * 75)
    print(report_text)
